In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [4]:

df = pd.read_csv("spotify_tracks.csv")

df["track_name"] = df["track_name"].str.lower().str.strip()
df["album_name"] = df["album_name"].str.lower().str.strip()
df["artist_name"] = df["artist_name"].str.lower().str.strip()
df["language"] = df["language"].str.lower().str.strip()


df = df.drop_duplicates(subset=["track_id"])

print(f"Dataset loaded: {len(df)} tracks")

Dataset loaded: 62239 tracks


In [5]:

features = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

In [10]:

print('Enter the song name you want recommendations for:')
SONG_NAME   = input('Song name : ').strip()
ARTIST_NAME = input('Artist name (optional, press Enter to skip): ').strip()

if ARTIST_NAME:
    song_match = df[
        (df["track_name"] == SONG_NAME) &
        (df["artist_name"] == ARTIST_NAME)
    ]
else:
    song_match = df[df["track_name"] == SONG_NAME]

if song_match.empty:
    print(" Song not found in dataset.")
else:
    chosen_song = song_match.iloc[0]
    print(" Selected:", chosen_song["track_name"], "-", chosen_song["artist_name"])

Enter the song name you want recommendations for:
 Selected: kun faya kun - a.r. rahman, javed ali, mohit chauhan


In [12]:
song_match = df[df["track_name"].str.contains(SONG_NAME, na=False)]

In [13]:
chosen_song = song_match.iloc[0]

chosen_track_id = chosen_song["track_id"]
chosen_track_name = chosen_song["track_name"]

print(f"✅ Selected: {chosen_track_name} (id: {chosen_track_id})")

✅ Selected: kun faya kun (from "rockstar") (id: 1zgucyNNKInroBqbZiNK0A)


In [15]:
matched = df[df["track_id"] == chosen_track_id]

if matched.empty:
    print("Song not found in the local dataset by track_id.")
    print("Trying by name...")
    
    matched = df[df["track_name"].str.lower() == chosen_track_name.lower()]

if matched.empty:
    print("❌ Song not in dataset. Try a different song.")
else:
    print(f"✅ Found in dataset: {matched.iloc[0]['track_name']} - {matched.iloc[0]['artist_name']}")

✅ Found in dataset: kun faya kun (from "rockstar") - a.r. rahman, javed ali, mohit chauhan


In [24]:
df = df.drop_duplicates(subset=["track_id"])


In [22]:
song_language = matched.iloc[0]["language"]

filtered_df = df[df["language"] == song_language]

In [27]:
song_vector = matched.iloc[[0]][features].values

song_language = matched.iloc[0]["language"]

filtered_df = df[df["language"] == song_language]

other_songs = filtered_df[
    filtered_df["track_name"] != matched.iloc[0]["track_name"]
].copy()

other_features = other_songs[features].values

sim_scores = cosine_similarity(song_vector, other_features)[0]

top_idx = np.argsort(sim_scores)[::-1][1:11]

recommendations = other_songs.iloc[top_idx][
    ["track_name", "artist_name", "album_name"]
].drop_duplicates(subset=["track_name"]).head(10).reset_index(drop=True)

print(f"\n🎧 Top 10 songs similar to '{chosen_track_name}' ({song_language}):\n")
print(recommendations.to_string(index=True))


🎧 Top 10 songs similar to 'kun faya kun (from "rockstar")' (hindi):

                                                    track_name                                                                                                                                          artist_name                                                   album_name
0  phir aur kya chahiye (from "zara hatke zara bachke") - lofi                                                                                     goldie khristi, arijit singh, sachin-jigar, amitabh bhattacharya  phir aur kya chahiye (from "zara hatke zara bachke") [lofi]
1                                      sun bhi le - chill trap                                                                                                  arijit singh, vishal mishra, raj shekhar, mindmaker                                      sun bhi le (chill trap)
2                                                    aa jao na                                                 